# Re-deriving the stored results from raw data

`reproduce_paper.ipynb` draws every figure and table from the derived data stored
in `results/` (reconstructions, statistics, cached simulations). This notebook goes
one level below: it decodes the raw hardware counts and shots again, re-runs the
noiseless simulations, and recomputes the cached simulation outputs, and compares
each result with the stored file. A final section explains how to build, submit and
decode a hardware package yourself.

Cells marked **slow** re-run the longer simulations (several minutes each). They are
skipped unless `RUN_SLOW = True`. With `RUN_SLOW = False` the notebook takes about
80 s (measured on a 4-core CPU).

In [1]:
import gzip
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import rgb_to_hsv
from PIL import Image

from tessa import (
    aer_sampler,
    apply_dynamic_range,
    data_path,
    decode,
    decode_from_cell_counts,
    decode_from_counts,
    encode,
    load_encoding,
)
from tessa_paper import (
    audio_midi,
    coherence,
    entanglement,
    genetic,
    pair_encoding,
    pixel_art,
)
from tessa_paper.paths import RESULTS

RUN_SLOW = False
START = time.perf_counter()
pd.set_option("display.width", 160)


def read_json_gz(path):
    with gzip.open(path, "rt") as f:
        return json.load(f)

## 1. QCAT: hardware counts -> reconstructions (Fig. 4, Table 1)

`results/hardware/qcat/counts/<device>_<mode>_<basis>.json.gz` has the whole-chip
counts returned by IBM (8192 shots per basis); `encodings.json` has the decode data
of each run. `decode_from_counts` inverts them.

In [2]:
QCAT = RESULTS / "hardware" / "qcat"
encodings = load_encoding(QCAT / "encodings.json")
stored = dict(np.load(QCAT / "reconstructions.npz"))
rows = []
for run, enc in encodings.items():
    dev, mode = run.split("/")
    bases = ["z"] if mode == "single" else ["z", "x"]
    counts = {
        b: read_json_gz(QCAT / "counts" / f"{dev}_{mode}_{b}.json.gz") for b in bases
    }
    raw = np.asarray(decode_from_counts(counts, enc))
    reference = (
        stored[f"{dev}/input_gray"]
        if mode == "single"
        else stored[f"{dev}/input_value_hue"][0]
    )
    value = raw if mode == "single" else raw[0]
    drr = apply_dynamic_range(value, reference, "mean_std")
    stored_drr = stored[f"{run}/hardware/mean_std"]
    rows.append(
        {
            "run": run,
            "shots per basis": sum(counts["z"].values()),
            "max |raw - stored|": np.abs(raw - stored[f"{run}/hardware/none"]).max(),
            "max |DRR - stored|": np.abs(
                drr - (stored_drr if mode == "single" else stored_drr[0])
            ).max(),
            "value MAE raw": np.abs(value - reference).mean(),
            "value MAE DRR": np.abs(drr - reference).mean(),
        }
    )
pd.DataFrame(rows)

,run,shots per basis,max |raw - stored|,max |DRR - stored|,value MAE raw,value MAE DRR
0,phoenix/single,8192,2.220446e-16,5.551115e-16,0.129887,0.074741
1,phoenix/pair,8192,4.024558e-16,5.551115e-16,0.152162,0.154148
2,fez/single,8192,2.220446e-16,4.440892e-16,0.131244,0.077570
3,fez/pair,8192,3.885781e-16,4.440892e-16,0.143579,0.133528


## 2. QCAT: the simulations - Table 1

**Noiseless.** The input images are encoded again and every super-pixel is simulated alone (`halo=None`) with Aer, seed 1, 8192 shots per basis; the result is compared with `results/simulation/qcat/sim.npz`. Might differ from paper sim slightly. 

**Local noise.** The local-noise simulation was built from
the per-qubit calibration at the time of the run, which was not saved, so that
simulation cannot be repeated exactly. 

In [3]:
IMAGES = {
    "phoenix": ("qcat28.png", "qcat28_color.png"),
    "fez": ("qcat.png", "qcat_color.png"),
}
sim = dict(np.load(RESULTS / "simulation" / "qcat" / "sim.npz"))
rows = []
for run, enc in encodings.items():
    dev, mode = run.split("/")
    gray, colour = IMAGES[dev]
    if mode == "single":
        image = np.asarray(Image.open(data_path(gray)).convert("L"), float) / 255
        circuit, again = encode(enc.graph, image, value_range=(0, 1))
    else:
        hsv = rgb_to_hsv(
            np.asarray(Image.open(data_path(colour)).convert("RGB"), float) / 255
        )
        circuit, again = encode(
            enc.graph,
            hsv[..., 2],
            hsv[..., 0],
            mode="pair",
            clamp=0.2 * np.pi,
            clamp_b=0.0,
            value_range=(0, 1),
            value_range_b=(0, 1),
        )
    noiseless = decode(
        circuit, again, aer_sampler(enc.graph, seed=1), shots=8192, halo=None
    )
    local = decode_from_cell_counts(
        read_json_gz(
            RESULTS / "simulation" / "qcat" / "counts" / f"{dev}_{mode}_local_sim.json.gz"
        ),
        enc,
    )
    noiseless = [noiseless] if mode == "single" else list(noiseless)
    local = [local] if mode == "single" else list(local)
    rows.append(
        {
            "run": run,
            "encoding identical": again.to_dict() == enc.to_dict(),
            "max |noiseless - stored|": max(
                np.abs(f - sim[f"{run}/noiseless/{j}"]).max()
                for j, f in enumerate(noiseless)
            ),
            "max |local sim. - stored|": max(
                np.abs(f - sim[f"{run}/local_sim/{j}"]).max() for j, f in enumerate(local)
            ),
        }
    )
pd.DataFrame(rows)

,run,encoding identical,max |noiseless - stored|,max |local sim. - stored|
0,phoenix/single,True,2.220446e-16,2.220446e-16
1,phoenix/pair,True,1.158795e-15,5.551115e-16
2,fez/single,True,2.220446e-16,2.220446e-16
3,fez/pair,True,8.465451e-16,3.330669e-16


## 3. Pixel art: raw shots -> decoded images (Fig. 6)

Both hardware packages are decoded again from `job/result.json` (the IBM
`PrimitiveResult` with every shot), `manifest.json` and `requests/`, into a temporary
folder, with `tessa.hardware.decode_folder` - the function behind
`python -m tessa.hardware decode <package> --out <folder>`. Every regenerated
`result.json` is compared with the stored one.

In [4]:
pixel_art.redecode_check()

,package,label,"structure, strings and images identical",floats differing,max relative float difference
0,PixelArt,anime,True,3,2.362957e-16
1,PixelArt,apple_moth_color,True,0,0.000000e+00
2,PixelArt,apple_moth_gray,True,4,3.272454e-16
3,PixelArt,cat_transition,True,3,3.845137e-16
4,PixelArt,citypopcat,True,4,2.452545e-16
5,PixelArt,sleepingCat,True,5,3.204536e-16
6,PixelArt,spaceship,True,0,0.000000e+00
7,PixelArt,statue,True,1,1.120459e-16
8,Pixels2,forest,True,4,2.175546e-16
9,Pixels2,ramen,True,4,2.178314e-16


## 4. Coherences: raw counts -> pair values (Fig. 9)

The measured K of all 1512 pixel pairs of experiment B, recomputed from the raw
counts (`results/hardware/coherence/counts/phoenix.json.gz`) and the experiment
design (`design.json`: which pair sat on which coupler in which circuit).

In [5]:
pairs = coherence.rederive_pairs_from_counts()
print(f"{len(pairs)} pairs, largest |K re-derived - K stored|: {pairs['|diff|'].max()}")
pairs.head()

1512 pairs, largest |K re-derived - K stored|: 0.0


,direction,pixel_i,pixel_j,value_i,value_j,data_qubit,address_qubit,K_exact,hardware_measured,hardware_measured_se,hardware_coupler_reference,local_sim_measured,local_sim_measured_se,local_sim_coupler_reference,K_rederived,|diff|
0,h,"0,0","0,1",0.396078,0.396078,91,101,1.0,0.998291,0.000646,0.993652,0.989258,0.001615,0.986084,0.998291,0.0
1,h,"0,1","0,2",0.396078,0.396078,35,45,1.0,0.975098,0.002450,0.964111,0.977295,0.002341,0.975098,0.975098,0.0
2,h,"0,2","0,3",0.396078,0.396078,111,110,1.0,0.996826,0.000880,0.985107,0.995117,0.001090,0.991455,0.996826,0.0
3,h,"0,3","0,4",0.396078,0.396078,86,96,1.0,0.998047,0.000690,0.988770,0.995605,0.001035,0.991455,0.998047,0.0
4,h,"0,4","0,5",0.396078,0.396078,71,72,1.0,0.998535,0.000598,0.990234,0.996826,0.000880,0.992432,0.998535,0.0


## 5. Genetic sequences: raw shots -> symbols (Table 8)

`results/hardware/genetic/<run>.npz` hold the packed shots of every circuit
(32768 per circuit); the ticket `<run>.json` holds the graph and the slot order used
at submission, and `constellation_points.npz` the constellation point sets of the
constellation runs. The sequences are re-encoded, the shots converted to counts with
`qiskit.primitives.BitArray` and decoded with `tessa.symbols_from_counts` for every
DRR method. The hardware accuracies are compared with the table the paper prints.

In [6]:
hw = genetic.hardware_results()
table = genetic.table_genetic(hw=hw)
hw_columns = [c for c in table.columns if c.startswith("hw_")]
table[hw_columns].round(1)

,,hw_raw,hw_min_max,hw_best,hw_best_method
sequence,encoding,,,,
POLG2 mRNA,constellation,83.8,84.3,84.3,min_max
SUR1 isoform 2,constellation,21.0,21.5,22.6,percentile
MT-CO3,constellation,92.1,91.5,92.3,min_avg_max
RRM1,constellation,69.9,69.3,69.9,mean_std
pikachurin,constellation,24.0,22.2,25.8,mean_std
Sonic hedgehog,constellation,73.4,73.2,76.2,min_avg_max
p53,constellation,70.5,70.0,71.5,mean_std
POLG2 mRNA,angle,57.0,47.5,68.5,mean_std
SUR1 isoform 2,angle,11.3,11.4,17.4,mean_std


In [7]:
paper_tex = (genetic.PAPER_TEX / "results_table.tex").read_text()
print(
    "regenerated results_table.tex identical to the paper's:",
    genetic.table_genetic_tex(table) == paper_tex,
)

regenerated results_table.tex identical to the paper's: True


## 6. Entanglement (Fig. 7, Fig. 8, Table 3)

The exact statevector computations behind App. C compared to saved info.

In [8]:
fresh = entanglement.compute_all(write=False)
cached = {
    "entanglement_map": entanglement.qcat_entanglement_map(),
    "area_law": entanglement.area_law(),
    "pairs": entanglement.pair_statistics(),
}
rows = []
for name, arrays in fresh.items():
    for key, arr in arrays.items():
        old = np.asarray(cached[name][key])
        numeric = arr.dtype.kind in "fc"
        rows.append(
            {
                "cache": name,
                "array": key,
                "shape": arr.shape,
                "identical": np.array_equal(arr, old, equal_nan=numeric),
                "max |diff|": np.nanmax(np.abs(arr - old))
                if numeric and arr.size
                else None,
            }
        )
pd.DataFrame(rows)

,cache,array,shape,identical,max |diff|
0,entanglement_map,image,"(21, 21)",True,0.000000e+00
1,entanglement_map,K_h,"(21, 21)",True,0.000000e+00
2,entanglement_map,K_v,"(21, 21)",True,0.000000e+00
3,entanglement_map,S,"(21, 21)",True,0.000000e+00
4,area_law,rows,"(4,)",True,NaN
5,area_law,cols,"(4,)",True,NaN
6,area_law,seeds,"(12,)",True,NaN
7,area_law,S_rows,"(4, 12)",False,1.287859e-14
8,area_law,S_cols,"(4, 12)",False,5.329071e-15
9,pairs,pairs,"(66, 2)",True,NaN


## 7. Pair encoding (Fig. 11 and the experiments behind App. E)

The original analysis script of the paper (not published).

In [9]:
reruns = {
    "e1_exact": pair_encoding.rerun_e1(),
    "e2_clamp": pair_encoding.rerun_e2(),
    "e3_colour": pair_encoding.rerun_e3(),
    "e4_sign_bit": pair_encoding.rerun_e4(),
}
pd.DataFrame(
    [
        {
            "experiment": key,
            "numbers compared": len(df),
            "max |diff|": df["abs_diff"].max(),
        }
        for key, df in ((k, pair_encoding.compare_rerun(k, v)) for k, v in reruns.items())
    ]
)

,experiment,numbers compared,max |diff|
0,e1_exact,20,1.110223e-16
1,e2_clamp,147,2.775558e-17
2,e3_colour,29,2.220446e-16
3,e4_sign_bit,11,1.734723e-18


## 8. MIDI and wavetable simulations (Table 6 pitch columns, Table 7, Fig. 12)

The FakeFez noise-model simulation (about 15 s), compared with the cache in
`results/simulation/midi_audio/`. These are simulations; the paper labels the MIDI
numbers as hardware (see REPRODUCIBILITY.md). The cache was written on Linux aarch64, where the
re-run is bit-identical; on other platforms the last bits of some numbers can
differ (on x86_64 at most 6e-19 here, and up to 2.8e-17 in section 7).

In [10]:
fresh = audio_midi.compute_all(write=False)
stored_midi = audio_midi.midi_results()["runs"]
midi_diff = max(
    abs(value - stored_midi[run][key])
    for run, report in fresh["midi"]["runs"].items()
    for key, value in report.items()
)
print(f"MIDI results: max |diff| from midi_fakefez.json = {midi_diff}")
cached = audio_midi.wavetable()
for key, arr in fresh["wavetable"].items():
    print(
        f"wavetable {key}: max |diff| = {np.abs(np.asarray(arr) - np.asarray(cached[key])).max()}"
    )

MIDI results: max |diff| from midi_fakefez.json = 6.398979969020694e-19
wavetable sine: max |diff| = 0.0
wavetable noiseless: max |diff| = 4.440892098500626e-16
wavetable fakefez: max |diff| = 4.440892098500626e-16


## 9. Slow: the simulated columns of Table 8 and the shot sweep

`genetic.rerun_simulation()` re-runs the "no noise" and "simulated noise" columns of
Table 8 (7 sequences x 2 encodings x 2 samplers, about 3.5 min on a 4-core VM) on the
ibm_phoenix calibration snapshot shipped with the library. On the machine used for
the reproduction, 27 of the 28 entries equal `results/simulation/genetic/metrics.json`;
in MT-CO3 / constellation / simulated noise one symbol of 784 differs (stored as
`metrics_rerun.json`, see REPRODUCIBILITY.md). The shot sweep
(`genetic.rerun_shot_sweep()`, about 15 min) regenerates `shot_sweep.json`.

In [11]:
def accuracy_table(metrics):
    return pd.DataFrame(
        {
            (key, entry): {m: v["accuracy"] for m, v in result["metrics"].items()}
            for key, entries in metrics.items()
            for entry, result in entries.items()
            if isinstance(result, dict) and "metrics" in result
        }
    ).T


if RUN_SLOW:
    rerun = genetic.rerun_simulation()
    stored_metrics = json.loads(
        (RESULTS / "simulation" / "genetic" / "metrics.json").read_text()
    )["sequences"]
    diff = (accuracy_table(rerun) - accuracy_table(stored_metrics)).abs()
    display(diff[diff.max(axis=1) > 0])
else:
    print("skipped (RUN_SLOW = False)")

skipped (RUN_SLOW = False)


In [12]:
if RUN_SLOW:
    sweep = genetic.rerun_shot_sweep()
    stored_sweep = json.loads(
        (RESULTS / "simulation" / "genetic" / "shot_sweep.json").read_text()
    )
    for key, entry in sweep.items():
        same = entry["errors"] == stored_sweep[key]["errors"]
        print(f"{key}: identical to shot_sweep.json: {same}")
else:
    print("skipped (RUN_SLOW = False)")

skipped (RUN_SLOW = False)


In [13]:
print(f"Run time of this notebook: {time.perf_counter() - START:.0f} s")

Run time of this notebook: 163 s


## 10. Running on IBM hardware yourself

The pixel-art runs of the paper were made with the hardware-package tool of the
library, `python -m tessa.hardware` (also installed as `tessa-hardware`; needs the
`ibm` extra). A package is a TOML file that lists the device, the shots and a set of
runs (images, representation, measurement bases, pixel-to-qubit mapping); the stored
`results/hardware/pixel_art/*/package.toml` files are examples (their image paths
refer to the original folder layout, see `data/README.md`). Subcommands:

| command | what it does |
|---|---|
| `python -m tessa.hardware check <package.toml>` | builds every circuit offline and reports qubits, depths and shots; nothing is submitted |
| `python -m tessa.hardware run <package.toml>` | builds, transpiles with the identity layout, submits one SamplerV2 job, waits, fetches the result and decodes it into a folder named after the package |
| `python -m tessa.hardware resume <folder>` | fetches the result of a submitted job later (for example after the process was stopped) |
| `python -m tessa.hardware decode <folder> [--out DIR]` | decodes a finished folder offline from `manifest.json`, `requests/` and `job/result.json` |
| `python -m tessa.hardware snapshot <backend> <file>` | saves the calibration snapshot of a device for offline noise models |

A minimal package, saved as `my_package.toml` in the repository root (image paths
are relative to the package file):

```toml
name = "MyPackage"
device = "phoenix"
shots = 8192
mapping = "lines"
seed = 0

[[runs]]
label = "qcat"
image = "data/images/qcat/qcat28.png"
representation = "gray"
basis = "z"
```

`python -m tessa.hardware check my_package.toml` then reports one circuit with 784
pixels on the 792 slots of the ibm_phoenix snapshot, and sends nothing.

Credentials are never read from a package file (a package that contains a token is
refused). Set them in the environment, or save them once with
`QiskitRuntimeService.save_account`:

```
export QISKIT_IBM_TOKEN=...       # your IBM Quantum API key
export QISKIT_IBM_INSTANCE=...    # your instance (CRN), optional if the saved account has one
python -m tessa.hardware run my_package.toml
```

The output folder holds a copy of the package, `manifest.json` (job id, backend,
circuits, run layout; no instance), `requests/`, `job/result.json` (every shot) and
the decoded `results/<label>/result.json`; this is the structure of the folders in
`results/hardware/pixel_art/`. Do not commit files that contain a token or an
instance CRN. In this repository `run`, `resume` and `snapshot` are tested only
against a simulated IBM service; `check` and `decode` run offline.

The same can be done from Python with the library's three calls, as in
`quickstart.ipynb`: `ibm_sampler(name="ibm_fez")`, `encode`, and
`decode(..., per_group=False)`. To keep the raw data, run the circuits returned by
`tessa.measurement_circuits(circuit, enc)` with your own sampler, save the counts,
and decode them later with `tessa.decode_from_counts(counts, enc)`; save the
encoding with `tessa.save_encoding`.